# 📘 Day 31 — KNN: Implementation & Scaling Impact

**Author:** Sahil-K-Y 
**Phase:** 3 — Tree Models & SVM 
**Date:** Day 031 of 214-Day AI/ML Roadmap

---
## 📌 Overview

Today we go deeper into KNN implementation details — building proper **sklearn Pipelines**, understanding the **dramatic impact of scaling**, exploring **distance-weighted KNN**, and applying KNN for **regression** tasks.

### 🎯 Learning Objectives
- Build **production-ready KNN pipelines** with scaling
- Deep dive into **StandardScaler vs MinMaxScaler** impact
- Implement **distance-weighted KNN** (`weights='distance'`)
- Apply **KNeighborsRegressor** for continuous predictions
- Use **GridSearchCV** for comprehensive hyperparameter tuning

---
## 🧠 1. KNN Pipeline Best Practices

### Why Pipelines Matter for KNN
KNN is **extremely sensitive to feature scales**. Using a Pipeline ensures:
- Scaling is applied **consistently** to train and test data
- No **data leakage** — scaler fits only on training data
- Clean, reproducible code

```python
# The correct way:
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=5))
])
pipe.fit(X_train, y_train)  # scaler fits on X_train only
pipe.predict(X_test)        # scaler transforms X_test using train stats
```

### Common Mistake
```python
# WRONG — data leakage!
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  # fits on ALL data including test
X_train, X_test = train_test_split(X_scaled)
```

---
## 🧠 2. StandardScaler vs MinMaxScaler

| Scaler | Formula | Output Range | Best When |
|--------|---------|-------------|----------|
| **StandardScaler** | (x - mean) / std | Unbounded (centered at 0) | Gaussian-like features, outliers present |
| **MinMaxScaler** | (x - min) / (max - min) | [0, 1] | Bounded features, no extreme outliers |

### Impact on KNN:
- Without scaling: Features with **large ranges dominate** distance
- StandardScaler: Each feature contributes **equally** to distance
- MinMaxScaler: All features in **same bounded range**

### ⚠️ Which to Choose?
- **StandardScaler** is generally safer (handles outliers better)
- **MinMaxScaler** can be better when feature distributions are **not Gaussian**
- Always **compare both** with cross-validation

---
## 🧠 3. KNN for Regression

KNN can also predict **continuous values** using `KNeighborsRegressor`:

### How it works:
- Find k nearest neighbors
- **Uniform:** Take the **mean** of neighbors' target values
- **Distance-weighted:** Take the **weighted mean** (closer neighbors have more influence)

### Key Differences from Classification:
| Aspect | Classification | Regression |
|--------|---------------|------------|
| Prediction | Majority vote | Mean of neighbors |
| Metric | Accuracy, F1 | MSE, MAE, R² |
| Edge case | Ties possible | Always gives value |
| Smoothness | Discrete boundaries | Continuous but step-like |

---
## 🧠 4. GridSearchCV for KNN Tuning

Multiple hyperparameters to tune simultaneously:

```python
param_grid = {
    'knn__n_neighbors': [3, 5, 7, 9, 11, 15, 21],
    'knn__weights': ['uniform', 'distance'],
    'knn__metric': ['euclidean', 'manhattan', 'minkowski'],
    'knn__p': [1, 2, 3]  # only used with minkowski
}
```

### Tips:
- Use **StratifiedKFold** for imbalanced data
- Set `n_jobs=-1` for parallel processing
- `cv=5` is standard for most datasets

---
## 💻 Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     GridSearchCV, StratifiedKFold)
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, mean_squared_error, r2_score)
from sklearn.datasets import fetch_openml

import warnings
warnings.filterwarnings('ignore')

plt.style.use('ggplot')
sns.set_palette('Set2')
print('✅ All libraries imported successfully!')

---
## 📊 Dataset: Glass Identification (UCI)

The **Glass Identification** dataset contains chemical composition of **214 glass samples** classified into **6 types** (building windows, vehicle windows, containers, tableware, headlamps).

**Features:** Refractive Index, Na, Mg, Al, Si, K, Ca, Ba, Fe 
**Target:** 6 types of glass

**Why this dataset?** Features have **very different scales** (RI ~1.5 vs Ca ~8.9 vs Fe ~0.05), making it perfect to demonstrate **scaling impact on KNN**.

In [ ]:
# --- Load Glass Identification Dataset ---
glass = fetch_openml('glass', version=1, as_frame=True, parser='auto')
df = glass.frame

print(f'Shape: {df.shape}')
print(f'\nFeature names: {list(glass.feature_names)}')
print(f'\nTarget classes: {df[glass.target_names[0]].value_counts().sort_index().to_dict()}')
print(f'\nFeature ranges (shows why scaling matters):')
print(df.describe().loc[['min', 'max']].T)
df.head()

---
## ✏️ Exercise 1: Scaling Impact Experiment
Quantify how scaling changes KNN performance.

**Tasks:**
1. Split data 80/20
2. Train KNN (k=5) **without** scaling → accuracy
3. Train KNN with **StandardScaler** → accuracy
4. Train KNN with **MinMaxScaler** → accuracy
5. Print all three and compute % improvement

In [ ]:
# YOUR CODE HERE


---
## ✏️ Exercise 2: KNN Pipeline with GridSearchCV
Build a Pipeline and tune all KNN hyperparameters.

**Tasks:**
1. Create Pipeline: StandardScaler → KNN
2. Define param_grid for n_neighbors, weights, metric
3. Run GridSearchCV with 5-fold StratifiedKFold
4. Print best params and best score

In [ ]:
# YOUR CODE HERE


---
## ✏️ Exercise 3: KNN Decision Boundary Visualization
Visualize how KNN creates boundaries (2D projection).

**Tasks:**
1. Select 2 most important features
2. Train KNN on just those 2 features
3. Create a meshgrid and predict across it
4. Plot the decision boundary with training points overlaid

In [ ]:
# YOUR CODE HERE


---
## ✏️ Exercise 4: KNN Regressor
Apply KNN for regression on a continuous target.

**Tasks:**
1. Use `Refractive Index (RI)` as target, rest as features
2. Train KNeighborsRegressor
3. Evaluate with MSE, MAE, R²
4. Plot actual vs predicted values

In [ ]:
# YOUR CODE HERE


---
## ✏️ Exercise 5: Curse of Dimensionality Experiment
Show how adding random noise features degrades KNN.

**Tasks:**
1. Start with original 9 features → record accuracy
2. Add 10 random noise columns → accuracy
3. Add 50 random noise columns → accuracy
4. Add 100 random noise columns → accuracy
5. Plot #features vs accuracy to visualize the curse

In [ ]:
# YOUR CODE HERE


---
## 📝 Key Takeaways
1. Scaling improvement (% gain): ...
2. Best hyperparams from GridSearch: ...
3. KNN Regressor R² score: ...
4. Curse of dimensionality impact: ...